# Imports

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import sklearn
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

from xgboost import XGBRegressor

In [ ]:
np.set_printoptions(precision=4, floatmode='fixed', suppress=True)

mpl.rcParams['figure.figsize'] = (10,5)

sklearn.set_config(transform_output="pandas")

# Objective

> Predict number of visits

# Load Data

In [ ]:
df = pd.read_csv("data/dataset.csv", header=0)

In [ ]:
df.info()

In [ ]:
df.head(10)

# Feature Engineering

In [ ]:
df['lag'] = df['num_visits_previous_day'].shift(1)

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.head()

> Move target column to the first position of the Dataframe

In [ ]:
col_to_move = df.pop('num_visits')
df.insert(loc=0, column='num_visits', value=col_to_move)

In [ ]:
df.head()

# Split Data

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=123)

In [ ]:
df_train.shape, df_test.shape

In [ ]:
X_train, y_train = df_train.drop(labels='num_visits', axis=1), df_train['num_visits']

In [ ]:
X_test, y_test = df_test.drop(labels='num_visits', axis=1), df_test['num_visits']

# XGBoost Model

In [ ]:
xgb_model = XGBRegressor(
    booster='gbtree',
    objective='reg:squarederror',
    gamma=10,
    eta=0.01,
    max_depth=10,
    subsample=0.5
)

In [ ]:
xgb_model.fit(X_train, y_train)

In [ ]:
y_test_pred = xgb_model.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

In [ ]:
print(f"RMSE: {rmse:.4f}")

In [ ]:
x = np.arange(len(X_test))

In [ ]:
plt.scatter(x=x,y=y_test, color='lightcoral', label=f"Ground truth (test)")
plt.scatter(x=x,y=y_test_pred, c='aqua',label=f"Predictions (test)")
plt.scatter(x=x[0], y=y_test_pred[:1], c='aqua', label=f"RMSE: {rmse:.4f}")
plt.legend()
plt.show()

# End